# Bài 1B — Train, Validation, Test Split & Phòng Chống Data Leakage

## 1. Mục Tiêu Bài Học
- **Phòng chống Data Leakage (Rò rỉ dữ liệu):** Nhận diện và loại bỏ thuộc tính `duration` khỏi ma trận đặc trưng trước khi phân chia dữ liệu.
- **Mã hóa nhãn mục tiêu:** Chuyển đổi biến `y` (`no` / `yes`) sang định dạng số nhị phân (`0` / `1`).
- **Phân chia phân tầng (Stratified Splitting):** Chia tập dữ liệu thành 3 tập riêng biệt: **Train (70%)**, **Validation (15%)**, và **Test (15%)** với tỷ lệ nhãn dương được bảo toàn nguyên vẹn.
- **Kỷ luật Machine Learning nghiêm ngặt:** Thiết lập ranh giới bất khả xâm phạm đối với tập Test (*sealed test set*).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# 1. Tải dữ liệu từ thư mục data/raw/
data_path = Path("data/raw/bank-additional-full.csv")
if Path.cwd().name == "notebooks":
    data_path = Path("../data/raw/bank-additional-full.csv")

df = pd.read_csv(data_path, sep=";")
print(f"Kích thước tập dữ liệu ban đầu: {df.shape}")

Kích thước tập dữ liệu ban đầu: (41188, 21)


## 2. Loại Bỏ Thuộc Tính `duration` Để Chống Data Leakage

Thuộc tính `duration` cho biết thời lượng cuộc gọi gần nhất (tính bằng giây). Biến này chỉ được ghi nhận **sau khi cuộc gọi đã kết thúc**. 
Nếu mục tiêu là xây dựng mô hình dự đoán khả thi **trước khi gọi** nhằm ưu tiên danh sách khách hàng tiềm năng, việc đưa `duration` vào sẽ gây ra hiện tượng **Data Leakage** (rò rỉ dữ liệu tương lai vào hiện tại), khiến mô hình trở nên vô dụng trong thực tế.

In [2]:
# Tách biến mục tiêu y và ma trận đặc trưng X (loại bỏ y và duration)
X = df.drop(columns=["y", "duration"])
y = df["y"].map({"no": 0, "yes": 1})

print(f"Số lượng đặc trưng đầu vào X (sau khi loại bỏ duration): {X.shape[1]} cột")
print(f"Phân phối nhãn mục tiêu y (0: no, 1: yes):\n{(y.value_counts(normalize=True) * 100).round(2)}")

Số lượng đặc trưng đầu vào X (sau khi loại bỏ duration): 19 cột
Phân phối nhãn mục tiêu y (0: no, 1: yes):
y
0    88.73
1    11.27
Name: proportion, dtype: float64


## 3. Phân Chia Tập Dữ Liệu: Train (70%) - Validation (15%) - Test (15%)

Do nhãn bị mất cân bằng nghiêm trọng (~88.73% `0` và ~11.27% `1`), việc phân chia ngẫu nhiên thuần túy (Random Split) có thể làm lệch tỷ lệ nhãn giữa các tập.

Do đó, ta bắt buộc áp dụng **Stratified Splitting** (`stratify=...`):
1. **Bước 1:** Tách 15% dữ liệu làm tập **Test** độc lập (niêm phong cho đánh giá cuối cùng).
2. **Bước 2:** Với 85% còn lại, tách tiếp 15% trên tổng (tương đương $\frac{15}{85} \approx 17.65\%$) làm tập **Validation**, còn lại 70% làm tập **Train**.

In [3]:
from sklearn.model_selection import train_test_split

# Bước 1: Tách tập Test độc lập (15% tổng dữ liệu)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Bước 2: Tách tập Train (70%) và Validation (15% tổng dữ liệu)
val_ratio_from_temp = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio_from_temp, random_state=42, stratify=y_temp
)

## 4. Kiểm Tra Tính Toàn Vẹn & Phân Phối Sau Khi Phân Chia

In [4]:
from IPython.display import display

# Lập bảng tổng hợp kích thước và tỷ lệ nhãn
split_summary = pd.DataFrame({
    "Số lượng mẫu": [len(X_train), len(X_val), len(X_test)],
    "Tỷ lệ tập (%)": [
        len(X_train) / len(df) * 100,
        len(X_val) / len(df) * 100,
        len(X_test) / len(df) * 100
    ],
    "Số lượng nhãn 1 (yes)": [y_train.sum(), y_val.sum(), y_test.sum()],
    "Tỷ lệ nhãn 1 (%)": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100
    ]
}, index=["Train Set", "Validation Set", "Test Set"])

split_summary["Tỷ lệ tập (%)"] = split_summary["Tỷ lệ tập (%)"].round(2)
split_summary["Tỷ lệ nhãn 1 (%)"] = split_summary["Tỷ lệ nhãn 1 (%)"].round(2)

display(split_summary)

# Kiểm tra không có sự trùng lặp chỉ số (Index Intersection)
train_idx = set(X_train.index)
val_idx = set(X_val.index)
test_idx = set(X_test.index)

assert len(train_idx.intersection(val_idx)) == 0, "Trùng lặp giữa Train và Val!"
assert len(train_idx.intersection(test_idx)) == 0, "Trùng lặp giữa Train và Test!"
assert len(val_idx.intersection(test_idx)) == 0, "Trùng lặp giữa Val và Test!"
assert len(X_train) + len(X_val) + len(X_test) == len(df), "Tổng số dòng không khớp dữ liệu gốc!"

print("[OK] Đã xác thực thành công: Không có rò rỉ hay trùng lặp chỉ số giữa các tập!")

,Số lượng mẫu,Tỷ lệ tập (%),Số lượng nhãn 1 (yes),Tỷ lệ nhãn 1 (%)
Train Set,28830,70.0,3248,11.27
Validation Set,6179,15.0,696,11.26
Test Set,6179,15.0,696,11.26


[OK] Đã xác thực thành công: Không có rò rỉ hay trùng lặp chỉ số giữa các tập!


## 5. Kỷ Luật Machine Learning Cho Các Giai Đoạn Tiếp Theo

1. **Tập Test được niêm phong hoàn toàn:** Tuyệt đối không dùng `X_test` và `y_test` trong quá trình huấn luyện, lựa chọn mô hình hay tinh chỉnh siêu tham số.
2. **Ngăn chặn rò rỉ dữ liệu khi tiền xử lý (Preprocessing Leakage):** Bất kỳ phép biến đổi nào (Imputation, Scaling, One-Hot Encoding) phải được **fit** trên tập Train, sau đó dùng transformer đó để **transform** trên Val và Test.